# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [18]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash"
)

## summarization middleware

Automatically summarize conversation history when approaching token limits , preserving recent messages wehile compressing older context . Summarixczation is useful for the following:
  
   * Long - runnign conversasations that exceed context window
   * Multi-turn dialogues with extensive history
   * Applications where preserving full conversation context matters.

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage , SystemMessage

### Message based Summarization

In [24]:

agent = create_agent(
    model = model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger=("messages",10),
            keep=("messages",4),
        )
    ]
)

In [25]:
### Run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [26]:
## Alternative test data 

questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 210/4?",
    "what is 15-6?",
    "what is 4*4?",
    "what is 25*25?",
    "what is 12*12?",
    

]

In [27]:
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messages: {len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='4ef8bde4-5353-4f96-a905-bf72c0d70c0b'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e529d-fb89-7513-9e52-66dc0109a6bd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 28, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})]}
Messages: 2
Messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='4ef8bde4-5353-4f96-a905-bf72c0d70c0b'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e529d-fb89-7513-9e52-66dc0109a6bd-0', tool_cal

## Token Size

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger=("tokens",550),
            keep=('tokens',200)
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

#token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars

In [30]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~375 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='80301384-f04e-44f5-8233-22430aeeae20'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'d3bc17f8-94c9-45b9-97d1-aa64136d9920': 'CqkCAQw51sdpAtKYv+/Jd2gKdX+XRhCbZyanLfUfwKLwxlM7ryoQFYdhefZnzHNcQ8dOccq3L2XbyK55G7AoM8LIQxT1VRtiTVWlU8f6NMbg3MbXNe9nlmeOeN5Swl+/4s6YFIxdM1zhPLN/DRR/lonrM+iTCXT4dVS13cRYiyUnwYxpoEKMn9ZYJmOORAxXKqDQiiw7cfB/MJoA9RycQQqlsmsmKyLAjVMywcz0HkPotzwwH4190PyK4lDxwqyGbPRGqBD1OncXB7cdXWLledt5yzls3AWZ9FzxmkV2x0VuDk7VxXDTec6Xdm+c8Eu/Dp1tTc48ptpe5e0fQPZ8DNurf+JE2M+p+H+vgy1sXo8v5kFFeoc36bEpaTzYA0ZZNTIGVJzqAdfCEz6i'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e52ab-53b5-71f0-8fee-b2949f19a4da-0', tool_calls=[{'name': 'search_hotel

## Fractions

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~36 tokens (0.0281%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='5f718c21-06f8-4fa9-97d8-52c17807fdbd'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'2e50dc60-03c4-4eb2-8364-b4b708accd37': 'CsUBAQw51sfjLTcoSfPanQtjqmigoRejWJPWLPFWrxVXTfljrNObOwtiG0IUGhpcabfaGtIHu6Gz3yf3htzHMJE0haxuV5eZW5BbMN/yxIGB6pwzx3C8MndhyouNDafA+ghkX5DhV0u9ZNV5Mo13DFbtn0WIc4/RO0VsyhTLFZLiFa+9tHsem3y8JB40t54/DkTL/oRFNnC7sA9jO6R8h/TcPPdTkdRpfy43TVBU361+XB9C1jqG7ZeL9xLSmb9L0TZGVQ92nUo='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e52ae-6612-7971-a792-2c61e2db1eb6-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '2e50dc60-03c4-4eb2-8364-b4b708accd37', 'type': 'tool_call'}], invalid_tool_calls=[], usage_met

# Human In the Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [33]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [34]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [35]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [36]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='d3b560b8-3f66-4c99-8208-3bf39649e4fe'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'d3267d97-799b-402f-8a7e-81fe9ec55bca': 'CoAEAQw51scZGPgBEIh9YOvanna0tFTHMn6fssD05ztF9vExRbi3SZu62ANZNrM4z6BkdjSgWv+1wHvCc0B7ShUhXqeDjhy+5XZwAqH4C2LnFsxFSQ3Z9fA6tuGcrMmCG/YV4JmlF8/8XvTws196siAdredhTC/aI5Bue5rdOftradH52P1uCXiOtJ2PTEJc0aVT6azpT6YXfN7S4hnARYWyPzJblH4TzwTaiwQi3failB1RDTI1+Zpcs281YDKhbQvYEYzgTDRUHIE7bq/7TeVyMbJNhGgj+S+7pDQlwUunfc79bLQRdHmn4biuzabD1T/ogZV9xwbuIL19Zc6iZOwiP99q9PxjniUKbemC/hcXSXs0HufmRV1c8y5APP5tK+27wMPUVAqXB2tiWYhfdtE8CBuDecl3oqXUrTOiU1Ep61ogvuzoLivZ+hr59CI12cWNANN8IDhWi8Qud94kya+COsv2H3g4GKznc7KiBIkQ8BFxiuolFNt1QC/AhoPkPW54MMHqXfoAwOAStIfr

In [37]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I have sent the email to john@test.com with the subject 'Hello' and body 'How are you?'.


In [38]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='d3b560b8-3f66-4c99-8208-3bf39649e4fe'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'d3267d97-799b-402f-8a7e-81fe9ec55bca': 'CoAEAQw51scZGPgBEIh9YOvanna0tFTHMn6fssD05ztF9vExRbi3SZu62ANZNrM4z6BkdjSgWv+1wHvCc0B7ShUhXqeDjhy+5XZwAqH4C2LnFsxFSQ3Z9fA6tuGcrMmCG/YV4JmlF8/8XvTws196siAdredhTC/aI5Bue5rdOftradH52P1uCXiOtJ2PTEJc0aVT6azpT6YXfN7S4hnARYWyPzJblH4TzwTaiwQi3failB1RDTI1+Zpcs281YDKhbQvYEYzgTDRUHIE7bq/7TeVyMbJNhGgj+S+7pDQlwUunfc79bLQRdHmn4biuzabD1T/ogZV9xwbuIL19Zc6iZOwiP99q9PxjniUKbemC/hcXSXs0HufmRV1c8y5APP5tK+27wMPUVAqXB2tiWYhfdtE8CBuDecl3oqXUrTOiU1Ep61ogvuzoLivZ+hr59CI12cWNANN8IDhWi8Qud94kya+COsv2H3g4GKznc7KiBIkQ8BFxiuolFNt1QC/AhoPkPW54MMHqXfoAwOAStIfr

## Reject

In [39]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [40]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [41]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I'm sorry, your request to send an email was rejected.


In [42]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='79d84916-684f-4d49-a719-f55a96971ebd'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'25863d7b-0cda-4517-8c44-91d680e93112': 'CrsFAQw51sdITzXsG1+T+HbQPf3f7aKCXisUV0faAHHKXx1UDxqkWS9pLx/9NmKLDGoX17ADDmEig3cHblLD5mevD461C20FPu4r3diZ/0AIYPgi9pZnBFGaD/P+kad73AirV7n3U+MsV1b8DW8ww3oNEmUMv+aMcoeoKKzFBtV3L7w9tGgaQY9DAOEdAw/xeUXjmEbi+V5xco6JDkIp4spis14MI84Ec/M85ZYggiOsfR+0nMOZyWXcysFHJb0AyMOKNuIJ2p1IsJw6A9Pc5sGaVPax0TtcKKUWDdprPq/upDNxnDb9repMeK+g7xT5zrfm6b75Eh+EVKOiuu6I7TQi3zTtZ2nPrlEEVuaetZFobWOGZdbhMPlCdmFmqbaTL3V0GJcMzqOs8bVEmxaVZ6r7+xeheFbFCDJwyA7C3b0R7hBqC0g8Nb5CtSl3ASuQwKixKGIa+fI9bvCorqiKd51kmQZ8CtAejm7GAafz5CGYV91bZEgpWMDxqXWsdKhUDTpOheEGTkW5imuoLFks

## Editing

In [44]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [45]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [46]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='bc95d4b1-376c-41c7-a531-6a64f15bbb5b'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Test", "body": "Hello", "recipient": "wrong@email.com"}'}, '__gemini_function_call_thought_signatures__': {'60e4376f-8588-49e0-ae4d-dc8bfb7761ad': 'CogEAQw51scyag/p1cmzThIfQfxWXVtYmOQ7jex95g9oVdzecq1Prc95Qhia4xpwQ+7F3AY6ljKsqVum++UIAtbe6P3ORuQ4IJys3b8TSDFGXGOl6Hor6V9MAVb3Nn44AgunICI/OQT6kSRhmyAzGdv3XJIXXbeKQ+SKGTC6NLuVH3VVkIZIA39/h8AsGdNTTBSIV+JYx9wJpgNQyVbFM7Nxggsxzeq5Vz9L43gB8/0HR8e/XufludRCBMtMwj+IKdlMBf0wn/f1nNQIyzrcO7W8UOy8F/AdpqXoIpwBaBBh0ktBXYiJ77dHyQdNcFJS4uOTeZ6lC3x92NYe6j9UOIlKgzZxkyotU63Hzqu/+z4AoP9Xyh/rwklopnqf1taV/NhPMxJ+YVegUAzMQIZUKvueLEs7k2Eor6hXz/fRMe0W4BB6xT4POTlvmD38OtnZtOOUC4V+l3p0hezjK2nzSs+VFIFofpnwyHBr4rNUQQl49hyRMIUACiNBbrYVRpcWWXZ6VvpUL0/0+w9RSXB2QkoW9zTKwiWV

In [47]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: I have sent the email to correct@email.com with the subject 'Corrected Subject' and body 'This was edited by human before sending'.


In [48]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='bc95d4b1-376c-41c7-a531-6a64f15bbb5b'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Test", "body": "Hello", "recipient": "wrong@email.com"}'}, '__gemini_function_call_thought_signatures__': {'60e4376f-8588-49e0-ae4d-dc8bfb7761ad': 'CogEAQw51scyag/p1cmzThIfQfxWXVtYmOQ7jex95g9oVdzecq1Prc95Qhia4xpwQ+7F3AY6ljKsqVum++UIAtbe6P3ORuQ4IJys3b8TSDFGXGOl6Hor6V9MAVb3Nn44AgunICI/OQT6kSRhmyAzGdv3XJIXXbeKQ+SKGTC6NLuVH3VVkIZIA39/h8AsGdNTTBSIV+JYx9wJpgNQyVbFM7Nxggsxzeq5Vz9L43gB8/0HR8e/XufludRCBMtMwj+IKdlMBf0wn/f1nNQIyzrcO7W8UOy8F/AdpqXoIpwBaBBh0ktBXYiJ77dHyQdNcFJS4uOTeZ6lC3x92NYe6j9UOIlKgzZxkyotU63Hzqu/+z4AoP9Xyh/rwklopnqf1taV/NhPMxJ+YVegUAzMQIZUKvueLEs7k2Eor6hXz/fRMe0W4BB6xT4POTlvmD38OtnZtOOUC4V+l3p0hezjK2nzSs+VFIFofpnwyHBr4rNUQQl49hyRMIUACiNBbrYVRpcWWXZ6VvpUL0/0+w9RSXB2QkoW9zTKwiWV